In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import minimize
from IPython.display import display
import rearrangement_algorithm as ra
from library.generate_marginals import CapacityMarginal

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
# generate marginals
seed = 0; rng = np.random.default_rng(seed)
d = 10
mu = rng.normal(loc=10, scale=4, size=d)
sd = rng.uniform(0.5, 5, size=d)
a = np.ones(d)
alpha = 0.9
eps = 0.1
scenario = 'ordinary'  # 'ordinary' or 'mixed'
outage_indices = [0, 1, 2, 3, 4]
p_outage = 0.05
rho = 0.5

def build_marginals(name):
    if name not in ('ordinary', 'mixed'):
        raise ValueError('Unknown scenario')
    if (len(outage_indices) != 5 or len(set(outage_indices)) != 5
            or any(not isinstance(i, (int, np.integer)) or not 0 <= i < d for i in outage_indices)):
        raise ValueError('Choose exactly five distinct asset indices in [0, 19]')
    return [CapacityMarginal(mu[i], sd[i],
             'outage' if name == 'mixed' and i in outage_indices else 'ordinary', p_outage)
            for i in range(d)]
    

scenario = "ordinary"
portfolios = {scenario: build_marginals(scenario)}
marginals = portfolios[scenario]

def marginal_table(ms):
    return pd.DataFrame({'Asset': np.arange(d), 'Type': [m.asset_type for m in ms],
        'Latent mean': mu, 'Latent SD': sd,
        'P90 capacity': [m.ppf(eps).item() for m in ms],
        'Probability at zero': [m.prob_zero for m in ms]})

for name, ms in portfolios.items():
    print(name)
    display(marginal_table(ms))

ordinary


,Asset,Type,Latent mean,Latent SD,P90 capacity,Probability at zero
0,0,ordinary,10.502921,4.171341,5.157132,5.903238e-03
1,1,ordinary,9.471581,0.512323,8.815012,1.301529e-76
2,2,ordinary,12.561691,4.358319,6.976280,1.974339e-03
3,3,ordinary,10.419600,0.651135,9.585137,6.165929e-58
4,4,ordinary,7.857323,3.783450,3.008637,1.891179e-02
5,5,ordinary,11.446380,1.290450,9.792602,3.655104e-19
6,6,ordinary,15.216000,4.384305,9.597287,2.596858e-04
7,7,ordinary,13.788324,2.936575,10.024951,1.330589e-06
8,8,ordinary,7.185059,1.848704,4.815850,5.084171e-05
9,9,ordinary,4.938314,2.402092,1.859909,1.989905e-02


Monte Carlo - solving the $b^*_{joint}$ and $b^*_{agg}$ from sample.

1. **Aggregate bid from Monte Carlo realizations**

Let $\omega_n$, $n=1,\ldots,N$, denote Monte Carlo realizations of the joint capacity vector, and define

$$
x_i^{(n)}=X_i(\omega_n),\qquad
z^{(n)}=\sum_{i=1}^{d}x_i^{(n)}.
$$

The P90 aggregate bid is the largest nonnegative bid delivered with probability at least $\alpha=0.9$:

$$
\boxed{
\begin{aligned}
b_{\mathrm{agg}}^*
&=\max_{b\ge0}\quad b\\
\text{s.t.}\quad
&\mathbb P\!\left(-\sum_{i=1}^{d}X_i(\omega)+b\le0\right)\ge0.9.
\end{aligned}
}
$$

Replace the probability by its empirical Monte Carlo frequency:

$$
\boxed{
\begin{aligned}
\widehat b_{\mathrm{agg},N}^*
&=\max_{b\ge0}\quad b\\
\text{s.t.}\quad
&\frac1N\sum_{n=1}^{N}
\mathbf1\!\left\{-\sum_{i=1}^{d}x_i^{(n)}+b\le0\right\}\ge0.9.
\end{aligned}
}
$$

Equivalently, using the requested scenario-set notation, let $\Omega_N=\{\omega_1,\ldots,\omega_N\}$ be the indexed sample set. Select a subset $S$ containing at least $\lceil0.9N\rceil$ sampled scenarios:

$$
\boxed{
\begin{aligned}
\widehat b_{\mathrm{agg},N}^*
&=\max_{b\ge0,\;S\subseteq\Omega_N}\quad b\\
\text{s.t.}\quad
&-\sum_{i=1}^{d}X_i(\omega)+b\le0,
&&\forall\,\omega\in S,\\
&|S|\ge\lceil0.9N\rceil.
\end{aligned}
}
$$

Thus $S$ is a selected set covering at least 90% of the samples. Requiring the inequality for **all** $N$ samples instead would give $b=\min_n z^{(n)}$, a 100% empirical-delivery requirement.

If $z_{(1)}\le\cdots\le z_{(N)}$ are the ordered aggregate samples and $m=\lceil0.9N\rceil$, the sample optimization has the exact solution

$$
\widehat b_{\mathrm{agg},N}^*=z_{(N-m+1)}.
$$

This order-statistic convention ensures at least 90% empirical coverage, including ties and atoms; an interpolated sample quantile need not satisfy that inequality. It estimates the lower-tail P90 bid under the **joint distribution used to generate the samples**, rather than optimizing over all dependence structures as in RA. Empirical coverage alone does not guarantee 90% population reliability.


In [4]:
cov_eq = lambda s, r: np.outer(s, s) * (np.full((len(s), len(s)), r) + (1 - r)*np.eye(len(s))) #sd_i sd_j rho

In [ ]:
mu_Z = a @ mu; sig_Z = np.sqrt(a @ cov_eq(sd, rho) @ a);
b_agg_G = mu_Z + stats.norm.ppf(eps) * sig_Z
b_agg_G

np.float64(78.01665539457882)

In [ ]:
mu_Z = a @ mu; sig_Z = np.sqrt(a @ cov_eq(sd, 0) @ a); b_agg_G = mu_Z + stats.norm.ppf(eps) * sig_Z
b_agg_G

np.float64(91.2229390385251)

In [33]:
rng = np.random.default_rng(seed + 1)
n_sim = 100000
rho = 1
R = np.full((d, d), rho) + (1 - rho)*np.eye(d)          
U = stats.norm.cdf(rng.multivariate_normal(np.zeros(d), R, size = n_sim))

X = np.column_stack([m.ppf(U[:, i]) for i, m in enumerate(marginals)])


alpha = 0.9; gamma = 0.01
N = X.shape[0]                              # X: (N, d) sampled capacities
z = np.sort(X.sum(axis=1))                  # ordered aggregates z_(1) <= ... <= z_(N)

m = int(stats.binom(N, alpha).ppf(1 - gamma)) + 1
b_agg_mc = z[N - m]                            # z_(N-m+1), 1-based
b_agg_mc

np.float64(69.30798945801956)

In [ ]:
qf = [m.ppf for m in marginals]
num_steps = 10000
wu, wo = ra.bounds_VaR(eps, qf, method = "lower", num_steps = num_steps, abstol = 1e-8)
b_agg_ra = wu[0]                            # lower end of RA bracket: safe bid
b_agg_ra

np.float64(57.70248855227046)

In [37]:
s_lo, s_hi  = ra.bounds_surv_probability(qf, b_agg_mc, num_steps = num_steps, abstol = 1e-8, method = 'lower')
s_lo[0], s_hi[0]

(0.762847900390625, 0.763031005859375)

In [ ]:
eps_w = eps + 3*np.sqrt(eps*(1 - eps)/n_sim)                 # mixed block slightly heavier than eps
(b_w, x_tail), _ = ra.bounds_VaR(eps_w, qf, method = 'lower', num_steps = num_steps, abstol = 1e-8)
p_body = eps_w + (1 - eps_w)*(np.arange(num_steps) + 0.5)/num_steps
x_body = np.column_stack([m.ppf(p_body) for m in marginals])

x_wc = np.vstack([x_tail, x_body]); 
w_wc = np.r_[np.full(num_steps, eps_w/num_steps), np.full(num_steps, (1 - eps_w)/num_steps)]
X = x_wc[rng.choice(len(x_wc), size = n_sim, p = w_wc)]
X

array([[10.66212981,  9.49113455, 12.72803598, ..., 13.90040508,
         7.25561912,  5.02999556],
       [ 9.88879092,  9.39615323, 11.92003257, ..., 13.35598351,
         6.91288178,  4.58466358],
       [18.98226656, 10.51301206, 21.42111866, ..., 19.75768448,
        10.94303428,  9.82119774],
       ...,
       [10.76733132,  9.50405538, 12.8379531 , ..., 13.97446572,
         7.30224355,  5.0905765 ],
       [12.15652277,  9.67467559, 14.28941435, ..., 14.95244033,
         7.91792161,  5.89055098],
       [11.65692112,  9.61331462, 13.76741831, ..., 14.60072658,
         7.69650233,  5.60285228]], shape=(100000, 10))

2. **Joint bid from Monte Carlo realizations (AND survival)**

Let $\omega_n$, $n=1,\ldots,N$, denote Monte Carlo realizations of the joint capacity vector, with $x_i^{(n)}=X_i(\omega_n)$. Assign a fixed commitment $c_i\ge0$ to each asset, with no redispatch. The bid is delivered only if **every** asset meets its own commitment:

$$
A_{\mathbf c}=\bigcap_{i=1}^{d}\{X_i\ge c_i\}.
$$

The P90 joint bid is the largest total commitment whose AND event holds with probability at least $\alpha=0.9$:

$$
\boxed{
\begin{aligned}
b_{\mathrm{joint}}^*
&=\max_{\mathbf c\ge0}\quad \sum_{i=1}^{d}c_i\\
\text{s.t.}\quad
&\mathbb P\!\left(-X_i(\omega)+c_i\le0,\ \ \forall i\right)\ge0.9.
\end{aligned}
}
$$

Replace the probability by its empirical Monte Carlo frequency:

$$
\boxed{
\begin{aligned}
\widehat b_{\mathrm{joint},N}^*
&=\max_{\mathbf c\ge0}\quad \sum_{i=1}^{d}c_i\\
\text{s.t.}\quad
&\frac1N\sum_{n=1}^{N}
\mathbf1\!\left\{-x_i^{(n)}+c_i\le0,\ \ \forall i\right\}\ge0.9.
\end{aligned}
}
$$

Equivalently, using the scenario-set notation, let $\Omega_N=\{\omega_1,\ldots,\omega_N\}$ be the indexed sample set. Select a subset $S$ containing at least $\lceil0.9N\rceil$ sampled scenarios:

$$
\boxed{
\begin{aligned}
\widehat b_{\mathrm{joint},N}^*
&=\max_{\mathbf c\ge0,\;S\subseteq\Omega_N}\quad \sum_{i=1}^{d}c_i\\
\text{s.t.}\quad
&-X_i(\omega)+c_i\le0,
&&\forall\,i,\ \forall\,\omega\in S,\\
&|S|\ge\lceil0.9N\rceil.
\end{aligned}
}
$$

For a given $S$, the best commitment is $c_i=\min_{\omega\in S}X_i(\omega)$, so the problem reduces to choosing which scenarios to keep. Requiring the inequality for **all** $N$ samples instead would give $c_i=\min_n x_i^{(n)}$ for each asset, a 100% empirical-delivery requirement.

Unlike the aggregate bid, there is no single order statistic here, because the kept set $S$ must work for all $d$ assets at once. With a binary $y_n=\mathbf 1\{\omega_n\in S\}$ and $\bar x_i=\max_n x_i^{(n)}$, it is the mixed-integer linear program

$$
\begin{aligned}
\max_{\mathbf c\ge0,\;\mathbf y\in\{0,1\}^N}\quad &\sum_{i=1}^{d}c_i\\
\text{s.t.}\quad
&c_i\le x_i^{(n)}+\big(\bar x_i-x_i^{(n)}\big)(1-y_n), &&\forall\,i,\ \forall\,n,\\
&\sum_{n=1}^{N}y_n\ge\lceil0.9N\rceil.
\end{aligned}
$$

Since $A_{\mathbf c}\subseteq\{\sum_iX_i\ge\sum_ic_i\}$, every feasible joint allocation is also feasible for the aggregate problem on the same samples, so $\widehat b_{\mathrm{joint},N}^*\le\widehat b_{\mathrm{agg},N}^*$. The difference $\Delta\widehat b=\widehat b_{\mathrm{agg},N}^*-\widehat b_{\mathrm{joint},N}^*\ge0$ is the bid lost by prohibiting redispatch. As for the aggregate bid, this estimates the AND reliability under the **joint distribution used to generate the samples**, and empirical coverage alone does not guarantee 90% population reliability.


In [45]:
from scipy.optimize import minimize

N = X.shape[0]; m = int(np.ceil(alpha*N))
V = np.column_stack([mg.cdf(X[:, i]) for i, mg in enumerate(marginals)])    # X_i >= F_i^-1(q) <=> V_i >= q, zero atom included

w_of = lambda th: np.exp(th)/np.exp(th).sum()                                  # split of the failure budget
c_of = lambda th: (lambda s: np.array([mg.ppf(min(s*wi, 1.)).item() for wi, mg in zip(w_of(th), marginals)]))(
                   np.partition((V/w_of(th)).min(axis = 1), N - m)[N - m])      # s: largest scale with AND in >= m samples
res = minimize(lambda th: -c_of(th).sum(), np.zeros(d), method = 'Powell', options = dict(xtol = 1e-4, ftol = 1e-7, maxfev = 20000))
c_joint = c_of(res.x); b_joint_mc = c_joint.sum()


In [46]:
b_joint_mc

np.float64(56.295729540863086)